# ProxyML
In this demo, we'll look at how you can use [ProxyML](https://proxyml.ai/) to better understand your machine learning models.  We'll be training a local model to predict whether a potential client is a good or a bad credit risk, and using ProxyML's Python SDK to learn more about our model and our data.

In [43]:
import os
import proxyml
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import classification_report
from sklearn.ensemble import GradientBoostingClassifier

In [126]:
# Add your ProxyML API key to the operating environment.  Don't have a key? Grab one for free at https://proxyml.ai/ !
os.environ['PROXYML_API_KEY'] = '<INSERT YOUR KEY HERE>'

## About The Data
We'll be using OpenML's [credit-g](https://www.openml.org/search?type=data&sort=runs&id=31&status=active) (German Credit) dataset for our demo.

In [3]:
credit_data = fetch_openml(name='credit-g', version=1, as_frame=True)
data, target = credit_data.data, credit_data.target
train_data, test_data, train_target, test_target = train_test_split(data, target, test_size=0.1)
print(f"{len(train_data)} sample(s) train, {len(test_data)} sample(s) test")

900 sample(s) train, 100 sample(s) test


In [4]:
display(train_data)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
426,no checking,28,critical/other existing credit,radio/tv,2743,<100,>=7,4,male single,none,2,car,29,none,own,2,skilled,1,none,yes
848,<0,9,existing paid,radio/tv,1364,<100,4<=X<7,3,male single,none,4,real estate,59,none,own,1,skilled,1,none,yes
657,no checking,48,existing paid,radio/tv,10222,no known savings,4<=X<7,4,male single,none,3,car,37,stores,own,1,skilled,1,yes,yes
184,0<=X<200,18,critical/other existing credit,new car,884,<100,>=7,4,male single,none,4,car,36,bank,own,1,skilled,2,yes,yes
26,no checking,6,no credits/all paid,radio/tv,426,<100,>=7,4,male mar/wid,none,4,car,39,none,own,1,unskilled resident,1,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
218,<0,24,existing paid,furniture/equipment,3021,<100,1<=X<4,2,male div/sep,none,2,real estate,24,none,rent,1,unskilled resident,1,none,yes
999,0<=X<200,45,critical/other existing credit,used car,4576,100<=X<500,unemployed,3,male single,none,4,car,27,none,own,1,skilled,1,none,yes
525,0<=X<200,26,existing paid,used car,7966,<100,<1,2,male single,none,3,car,30,none,own,2,skilled,1,none,yes
385,no checking,18,critical/other existing credit,radio/tv,2238,<100,1<=X<4,2,female div/dep/mar,none,1,car,25,none,own,2,skilled,1,none,yes


In [6]:
display(train_target)

734    good
763     bad
235     bad
365    good
137     bad
       ... 
735    good
149    good
834     bad
84     good
826     bad
Name: class, Length: 900, dtype: category
Categories (2, str): ['bad', 'good']

### Feature Selection
We'll use a simple set of features based on the columns in the dataset: we'll one-hot encode the categorical columns, and use the numeric columns as-is.

#### Immutable Columns
Depending on the scenario, there may be some attributes of the data that shouldn't be used for prediction.  In many cases this will also mean that these same attributes shouldn't be used for explaining predictions.  ProxyML supports the exclusion of features by considering them as _immutable_: surrogate models will not use the columns marked as immutable.

For our credit data demo, it makes sense to not consider age or gender in risk assessment.  We'll set the corresponding columns as immutable.

In [5]:
# List of columns we should NOT use for predicting credit risk
immutable_cols = ['age', 'personal_status']  # personal_status encodes gender

# List of numeric cols
numeric_cols = ['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']
numeric_cols = list(set(numeric_cols) - set(immutable_cols))
display(numeric_cols)

['existing_credits',
 'duration',
 'installment_commitment',
 'credit_amount',
 'residence_since',
 'num_dependents']

In [6]:
# List of columns we'll one-hot encode
categorical_cols = [
    'checking_status', 
    'credit_history', 
    'purpose', 
    'savings_status', 
    'employment', 
    'personal_status', 
    'other_parties', 
    'property_magnitude', 
    'other_payment_plans', 
    'housing', 
    'job', 
    'own_telephone', 
    'foreign_worker'
]

categorical_cols = list(set(categorical_cols) - set(immutable_cols))
display(categorical_cols)

['own_telephone',
 'purpose',
 'housing',
 'checking_status',
 'property_magnitude',
 'job',
 'foreign_worker',
 'employment',
 'other_payment_plans',
 'other_parties',
 'savings_status',
 'credit_history']

In [7]:
# Construct our final "black box" feature set - these are the features that will be made available to our credit risk prediction model.

black_box_feature_cols = list()
black_box_feature_cols.extend(numeric_cols)
black_box_feature_cols.extend(categorical_cols)
display(black_box_feature_cols)

['existing_credits',
 'duration',
 'installment_commitment',
 'credit_amount',
 'residence_since',
 'num_dependents',
 'own_telephone',
 'purpose',
 'housing',
 'checking_status',
 'property_magnitude',
 'job',
 'foreign_worker',
 'employment',
 'other_payment_plans',
 'other_parties',
 'savings_status',
 'credit_history']

Now we'll construct a scikit-learn [ColumnTransformer](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html) pipeline that'll one-hot encode the `categorical_cols` and pass through the `numeric_cols`.

In [8]:
# Set up the ColumnTransformer
ct = ColumnTransformer(
    transformers=[
        ('encoder', OneHotEncoder(), categorical_cols)
    ],
    remainder='passthrough'  # Keeps the numeric columns unchanged
)

# Transform the data
X_train = ct.fit_transform(train_data[black_box_feature_cols])
X_test = ct.transform(test_data[black_box_feature_cols])

y_train = train_target
y_test = test_target

### Training The Local Model
We're ready to train our credit risk predictor.  We'll run a simple cross-validator over a small parameter space for [gradient boosting](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html).

In [9]:
black_box = RandomizedSearchCV(
    estimator=GradientBoostingClassifier(),
    param_distributions={
        'n_estimators': [100, 256, 512],
        'learning_rate': [0.05, 0.075, 0.1, 0.2],
        'subsample': [1.0, 0.75, 0.5],
        'max_features': [None, 'sqrt', 'log2'],
        'max_depth': [3, 2, 1],
    },
    n_jobs=-1
)
black_box.fit(X_train, y_train)

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",GradientBoostingClassifier()
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'learning_rate': [0.05, 0.075, ...], 'max_depth': [3, 2, ...], 'max_features': [None, 'sqrt', ...], 'n_estimators': [100, 256, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",10
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",None
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here.

In [11]:
print(f"Best model scored {black_box.best_score_:.2f} with these parameters: {black_box.best_params_}")
print(f"Score on test data: {black_box.score(X_test, y_test)}")

Best model scored 0.77 with these parameters: {'subsample': 0.5, 'n_estimators': 512, 'max_features': 'sqrt', 'max_depth': 1, 'learning_rate': 0.075}
Score on test data: 0.78


In [12]:
print("Black box classification report:")
print(classification_report(y_test, black_box.predict(X_test)))

Black box classification report:
              precision    recall  f1-score   support

         bad       0.57      0.48      0.52        25
        good       0.84      0.88      0.86        75

    accuracy                           0.78       100
   macro avg       0.70      0.68      0.69       100
weighted avg       0.77      0.78      0.77       100



## Getting Started With ProxyML
We've got our credit risk predictor, now we'd like to learn more about the predictions it makes.  ProxyML works by training a _surrogate model_ on the basic statistics of your data rather than the data itself, so your data never leaves your server.

### Schema Generation
Your data's statistics are shared with ProxyML by providing a JSON _schema_.  Depending on the size and nature of your data, the schema can be fairly complex but luckily we've got a few options to get you started:

* The ProxyML Python SDK's [`get_schema` function ](https://github.com/proxyml/proxyml-sdk-python/blob/df204c9518ad5135bca1aa9eb1b0dddf95549143/src/proxyml/schema.py#L39) can generate a schema from a pandas DataFrame;
* The [ProxyML Schema Generator page](https://proxyml.github.io/schema-builder/) accepts CSVs and TSVs and generates the schema in your browser; and
* The [ProxyML Schema Generator source code](https://github.com/proxyml/schema-builder) is also available if you'd prefer to run everything yourself.

We'll use the SDK for this demo.

**Note:**  you should consider auto-generated schema as a good starting point, but you should carefully review and adjust the information as needed to best represent your data. It's so important in fact that the SDK includes a note to this effect.

In [13]:
# Creating a ProxyML Schema
surrogate_schema = proxyml.get_schema(train_data, immutable_cols=immutable_cols)
if '_note' in surrogate_schema:
    print(f"Note: {surrogate_schema['_note']}\n")
for feature in surrogate_schema['features']:
    for k, v in feature.items():
        print(f"{k}: {v}")
    print()

Note: Auto-generated schema. Review and adjust types as needed. Integer columns default to count type - consider categorical_ordinal for ordered categories like ratings or class labels.

type: categorical
name: checking_status
valid_categories: {'no checking': 0.3877777777777778, '<0': 0.2788888888888889, '0<=X<200': 0.2722222222222222, '>=200': 0.06111111111111111}
immutable: False

type: count
name: duration
lambda: 20.953333333333333
max: 72
immutable: False

type: categorical
name: credit_history
valid_categories: {'existing paid': 0.53, 'critical/other existing credit': 0.29444444444444445, 'delayed previously': 0.09, 'all paid': 0.044444444444444446, 'no credits/all paid': 0.04111111111111111}
immutable: False

type: categorical
name: purpose
valid_categories: {'radio/tv': 0.28444444444444444, 'new car': 0.23222222222222222, 'furniture/equipment': 0.17444444444444446, 'used car': 0.10444444444444445, 'business': 0.09777777777777778, 'education': 0.05333333333333334, 'repairs': 0.

With the schema generated, we're ready to share it with ProxyML.  

In [14]:
schema_store_result = proxyml.put_schema(surrogate_schema)
print(schema_store_result)

{'features': [{'name': 'checking_status', 'immutable': False, 'valid_categories': {'no checking': 0.3877777777777778, '<0': 0.2788888888888889, '0<=X<200': 0.2722222222222222, '>=200': 0.06111111111111111}, 'type': 'categorical'}, {'name': 'duration', 'immutable': False, 'lambda': 20.953333333333333, 'max': 72, 'type': 'count'}, {'name': 'credit_history', 'immutable': False, 'valid_categories': {'existing paid': 0.53, 'critical/other existing credit': 0.29444444444444445, 'delayed previously': 0.09, 'all paid': 0.044444444444444446, 'no credits/all paid': 0.04111111111111111}, 'type': 'categorical'}, {'name': 'purpose', 'immutable': False, 'valid_categories': {'radio/tv': 0.28444444444444444, 'new car': 0.23222222222222222, 'furniture/equipment': 0.17444444444444446, 'used car': 0.10444444444444445, 'business': 0.09777777777777778, 'education': 0.05333333333333334, 'repairs': 0.024444444444444446, 'other': 0.011111111111111112, 'domestic appliance': 0.01, 'retraining': 0.00777777777777

## Data Synthesis
The next step in our process is to generate synthetic data, which are based on the descriptive statistics we just uploaded.  If we provide these synthetic data to our credit risk model, it can make risk predictions...and we can train a surrogate to predict its risk prediction by pairing the synthetic data with the risk predictions.  That's how ProxyML provides explainability without ever actually seeing your real data.

ProxyML's `synthesize_data` has two modes of operation: _neighbors_ and _blended_.  In neighbors (also known as "global") mode, synthesis is strictly based on a random sample of the synthetic data's featurespace.  For example, if your data has a continuous numeric feature `Retail Sq. Ft` with mean $\mu$ and standard deviation $\sigma$, every synthetic sample will have a `Retail Sq. Ft` feature drawn from `np.random.normal(loc=`$\mu$ `, scale=` $\sigma$ `)` .

In blended mode, synthesis is based on a blend of global sampling and local pertubations around a provided sample, provided in your call to `synthesize_data`.  This is particularly useful for "what if?" counterfactual scenarios - in our current credit risk problem for example we might be interested to see what would have to change for a `bad` risk to change to a `good` risk.  This does require sharing a sample with ProxyML, but it doesn't necessarily have to be a real sample - you could for example generate your own synthetic sample that was a good match for the real sample, or simply generate an initial batch of global synthetic samples with ProxyML and then choose the synthetic sample that most closely matches your real sample.

To keep things simple for this demo, we'll stick to global mode.

In [15]:
# Synthesize data
synth_df = proxyml.synthesize_data(num_points=500)
print(f"\nSynthesized {len(synth_df)} rows with columns: {list(synth_df.columns)}")
display(synth_df)


Synthesized 500 rows with columns: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker']


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,0<=X<200,22,existing paid,education,3325,100<=X<500,1<=X<4,0,female div/dep/mar,none,1,real estate,38,none,own,0,unskilled resident,2,none,yes
1,<0,19,existing paid,education,3341,>=1000,<1,3,male single,none,3,no known property,36,none,for free,0,skilled,0,none,yes
2,>=200,16,existing paid,business,3378,no known savings,1<=X<4,3,male single,none,2,real estate,30,none,rent,1,skilled,0,none,yes
3,<0,20,all paid,new car,3266,<100,>=7,3,female div/dep/mar,none,2,real estate,34,none,rent,3,skilled,0,none,yes
4,0<=X<200,15,existing paid,radio/tv,3253,<100,1<=X<4,4,female div/dep/mar,none,2,car,22,none,rent,1,unskilled resident,2,none,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,no checking,14,existing paid,retraining,3268,no known savings,>=7,4,male single,none,2,real estate,34,bank,own,0,skilled,2,none,yes
496,0<=X<200,21,critical/other existing credit,radio/tv,3283,>=1000,>=7,4,male single,none,2,car,29,none,rent,0,unskilled resident,1,none,yes
497,<0,15,existing paid,repairs,3375,<100,4<=X<7,3,female div/dep/mar,none,4,life insurance,42,none,own,0,skilled,1,yes,yes
498,no checking,26,critical/other existing credit,radio/tv,3348,100<=X<500,<1,4,male div/sep,none,1,real estate,31,none,own,2,unskilled resident,0,yes,yes


## Surrogate Model
We're ready to train our surrogate model! All we need are the synthetic data we just generated, and our credit risk model's predictions for these synthetic data.

In [17]:
# Train a surrogate model
black_box_predictions = black_box.predict(ct.transform(synth_df[black_box_feature_cols])).tolist()
train_result = proxyml.train_surrogate(
    samples=synth_df.values.tolist(),
    predictions=black_box_predictions,
    feature_names=list(synth_df.columns),
    task="classification",  # "classification," "regression," or "auto" in which case ProxyML will try to programatically determine the type of task
    test_size=0.2,
)
print("\nSurrogate training result:")
for k, v in train_result.items():
    print(f"{k}: {v}")


Surrogate training result:
version: 5b90955b-b5ba-4b82-9022-30325797675a
trained_at: 2026-04-22 13:29:39
task: classification
name: None
comments: None
feature_names: ['checking_status', 'duration', 'credit_history', 'purpose', 'credit_amount', 'savings_status', 'employment', 'installment_commitment', 'personal_status', 'other_parties', 'residence_since', 'property_magnitude', 'age', 'other_payment_plans', 'housing', 'existing_credits', 'job', 'num_dependents', 'own_telephone', 'foreign_worker']
metrics: {'f1': 0.9439257294429708, 'accuracy': 0.94}


We can retrieve additional details about our surrogate, for example a summary of our new surrogate:

In [18]:
surrogate_summary = proxyml.get_model_summary(version=train_result['version'])
surrogate_summary

{'model_version': '5b90955b-b5ba-4b82-9022-30325797675a',
 'task': 'classification',
 'trained_at': '2026-04-22 13:29:39',
 'name': None,
 'comments': None,
 'feature_names': ['checking_status',
  'duration',
  'credit_history',
  'purpose',
  'credit_amount',
  'savings_status',
  'employment',
  'installment_commitment',
  'personal_status',
  'other_parties',
  'residence_since',
  'property_magnitude',
  'age',
  'other_payment_plans',
  'housing',
  'existing_credits',
  'job',
  'num_dependents',
  'own_telephone',
  'foreign_worker'],
 'metrics': {'f1': 0.9439257294429708, 'accuracy': 0.94},
 'feature_importances': [{'feature': 'checking_status',
   'coefficient': 2.3572837484173776,
   'abs_coefficient': 2.3572837484173776},
  {'feature': 'credit_history',
   'coefficient': 1.745346950800371,
   'abs_coefficient': 1.745346950800371},
  {'feature': 'other_parties',
   'coefficient': 1.464086389976435,
   'abs_coefficient': 1.464086389976435},
  {'feature': 'employment',
   'coef

Or just the feature importances:

In [20]:
surrogate_importances = proxyml.get_feature_importances(version=train_result['version'])
surrogate_importances

{'feature_importances': [{'feature': 'checking_status',
   'coefficient': 2.3572837484173776,
   'abs_coefficient': 2.3572837484173776},
  {'feature': 'credit_history',
   'coefficient': 1.745346950800371,
   'abs_coefficient': 1.745346950800371},
  {'feature': 'other_parties',
   'coefficient': 1.464086389976435,
   'abs_coefficient': 1.464086389976435},
  {'feature': 'employment',
   'coefficient': 1.2924246499428986,
   'abs_coefficient': 1.2924246499428986},
  {'feature': 'installment_commitment',
   'coefficient': -1.14739846073121,
   'abs_coefficient': 1.14739846073121},
  {'feature': 'property_magnitude',
   'coefficient': 1.0134112604474887,
   'abs_coefficient': 1.0134112604474887},
  {'feature': 'purpose',
   'coefficient': 0.9081586677445836,
   'abs_coefficient': 0.9081586677445836},
  {'feature': 'duration',
   'coefficient': -0.8997301796707203,
   'abs_coefficient': 0.8997301796707203},
  {'feature': 'savings_status',
   'coefficient': 0.8461807162161756,
   'abs_coeffi

And of course we can make predictions with our surrogate, and compare with our credit risk model.  Note that the surrogate won't always agree with your "real" model right off the bat, you may need to to experiment with schema and synthetic data to get the best result.

In [40]:
sample_point = synth_df.sample()
xformed_sample = ct.transform(sample_point)
display(sample_point)

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
307,<0,20,existing paid,radio/tv,3425,>=1000,>=7,2,male single,none,3,life insurance,24,none,own,3,skilled,0,yes,yes


In [41]:
# Surrogate prediction
proxyml.predict(sample=xformed_sample.tolist()[0])

{'prediction': 'bad',
 'model_version': 'surrogate-5b90955b-b5ba-4b82-9022-30325797675a-classification'}

In [42]:
# Compare with our credit risk predictor
print(f"Credit risk prediction: '{black_box.predict(xformed_sample)[0]}', probabilities {black_box.predict_proba(xformed_sample)[0]}")

Credit risk prediction: 'good', probabilities [0.13654125 0.86345875]


## Counterfactuals
Let's take a look at counterfactuals in more detail.  Suppose we're interested in seeing what it would take to get some of the bad credit risks to become good credit risks.

In [113]:
xformed_synthetic_data = ct.transform(synth_df)
black_box_predictions = black_box.predict(xformed_synthetic_data)

bad_risk_samples = (
    synth_df[black_box_predictions == 'bad']
    .head(5)
    .apply(lambda x: x.map(lambda v: v.item() if hasattr(v, 'item') else v))
    .values.tolist()
)

Now we can send these predicted _bad_ risks to ProxyML, and have it return counterfactuals. We'll look at each feature of each sample and identify which feature(s) could be changed to potentially change the risk to _good_.

In [125]:
cfs = proxyml.find_counterfactuals(
    samples=bad_risk_samples,
    target='good',
    n_neighbors=10000,
    perturbation_scale=0.25
)

for i, cf in enumerate(cfs):
    sample_df = pd.DataFrame([bad_risk_samples[i]], columns=cf.columns)
    diff = sample_df.iloc[0].compare(cf.iloc[0], result_names=('Original', 'Counterfactual'))
    print(f"Sample {i + 1}:")
    display(diff)
    print(f"Original credit risk prediction: '{black_box.predict(ct.transform(sample_df))[0]}' ({black_box.predict_proba(ct.transform(sample_df))[0]})")
    print(f"Counterfactual credit risk prediction: '{black_box.predict(ct.transform(cf))[0]}' ({black_box.predict_proba(ct.transform(cf))[0]})")
    print()

Sample 1:


,Original,Counterfactual
purpose,education,radio/tv


Original credit risk prediction: 'bad' ([0.54923914 0.45076086])
Counterfactual credit risk prediction: 'good' ([0.40034835 0.59965165])

Sample 2:


,Original,Counterfactual
checking_status,<0,no checking


Original credit risk prediction: 'bad' ([0.73798513 0.26201487])
Counterfactual credit risk prediction: 'good' ([0.33862224 0.66137776])

Sample 3:


...

Original credit risk prediction: 'bad' ([0.81616345 0.18383655])
Counterfactual credit risk prediction: 'good' ([0.44660566 0.55339434])

Sample 4:


,Original,Counterfactual
credit_history,all paid,critical/other existing credit


Original credit risk prediction: 'bad' ([0.61161478 0.38838522])
Counterfactual credit risk prediction: 'good' ([0.35947546 0.64052454])

Sample 5:


,Original,Counterfactual
checking_status,<0,no checking


Original credit risk prediction: 'bad' ([0.61571314 0.38428686])
Counterfactual credit risk prediction: 'good' ([0.2255571 0.7744429])



Your results may vary, but for this particular run ProxyML found that changing a single feature for each of the _bad_ credit risks was sufficient, which was backed up by having our black box credit risk model run the prediction on the counterfactual.  Some general observations:

* Having no checking account is preferable to having a negative balance in the checking account.
* Credit history seems counterintuitive at first, but in this particular dataset `all paid` includes people with no credit history, suggesting that even a problematic history is better than no history at all.
* The confidence shifts seem reasonable, in the sense that none of them are overwhelming flips, they're all moving from ~60-80% bad to ~55-66% good.

Remember that none of these counterfactuals required access to the original training data or the internals of the black box model!  We only needed the surrogate model's approximation of the black box's behavior on synthetic data.